# 02 - ML De-anonymization Attacks

Plug-and-play red team: use scikit-learn sparse vectors, `NearestNeighbors`, and `LogisticRegression`. This notebook keeps the attack short and auditable.

Core references: Netflix de-anonymization, scikit-learn, nearest-neighbor search, logistic regression, and membership inference are cited in `notebooks/README.md`.

In [ ]:
from __future__ import annotations

import os
import sys
import tempfile
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "matplotlib-cache"))

import numpy as np
import pandas as pd
try:
    from scipy.sparse import csr_matrix, vstack
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import accuracy_score, roc_auc_score
    from sklearn.model_selection import train_test_split
    from sklearn.neighbors import NearestNeighbors
except ImportError as exc:
    raise ImportError("Install notebook ML deps with: python -m pip install -e '.[ml]'") from exc

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from guardrails_sensitive_data.anonymization import add_time_and_rating_features, make_releases
from guardrails_sensitive_data.netflix_io import read_netflix_ratings

DATA_DIR = ROOT / "data" / "netflix"
SEED = 333
MAX_ROWS = 250_000

In [ ]:
ratings = add_time_and_rating_features(read_netflix_ratings(DATA_DIR, max_rows=MAX_ROWS, include_date=True))
rng = np.random.default_rng(SEED)
users = ratings["customer_id"].drop_duplicates().to_numpy()
holdout_users = set(rng.choice(users, size=max(1, int(0.25 * len(users))), replace=False).astype(int))
release = ratings[~ratings["customer_id"].isin(holdout_users)].copy()
holdout = ratings[ratings["customer_id"].isin(holdout_users)].copy()
{"release_users": release["customer_id"].nunique(), "holdout_users": holdout["customer_id"].nunique()}

## Sparse vectors and nearest-neighbor linkage

Represent each user as a sparse `movie_id -> centered rating` vector. Cosine nearest neighbors become a one-line model fit.

In [ ]:
def fit_user_vectorizer(frame: pd.DataFrame):
    user_index = pd.Index(frame["customer_id"].drop_duplicates())
    movie_index = pd.Index(frame["movie_id"].drop_duplicates())
    rows = user_index.get_indexer(frame["customer_id"])
    cols = movie_index.get_indexer(frame["movie_id"])
    values = frame["rating"].to_numpy(dtype=np.float32) - 3.0
    matrix = csr_matrix((values, (rows, cols)), shape=(len(user_index), len(movie_index)))
    return user_index, movie_index, matrix

release_users, release_movies, release_matrix = fit_user_vectorizer(release)
nn = NearestNeighbors(metric="cosine", algorithm="brute", n_neighbors=10).fit(release_matrix)

In [ ]:
def sample_profiles(frame: pd.DataFrame, n_users: int = 300, n_facts: int = 6, seed: int = SEED) -> dict[int, pd.DataFrame]:
    rng = np.random.default_rng(seed)
    eligible = frame["customer_id"].value_counts().loc[lambda s: s >= n_facts].index.to_numpy()
    users = rng.choice(eligible, size=min(n_users, len(eligible)), replace=False)
    return {
        int(user): frame[frame["customer_id"] == user].sample(n_facts, random_state=int(rng.integers(2**31)))[["movie_id", "rating"]]
        for user in users
    }


def profile_vector(profile: pd.DataFrame, movie_index: pd.Index) -> csr_matrix:
    cols = movie_index.get_indexer(profile["movie_id"])
    keep = cols >= 0
    values = profile.loc[keep, "rating"].to_numpy(dtype=np.float32) - 3.0
    return csr_matrix((values, ([0] * keep.sum(), cols[keep])), shape=(1, len(movie_index)))


def nearest_neighbor_attack(profiles: dict[int, pd.DataFrame], top_k: int = 10) -> pd.DataFrame:
    rows = []
    for target_user, profile in profiles.items():
        vector = profile_vector(profile, release_movies)
        distances, positions = nn.kneighbors(vector, n_neighbors=top_k)
        candidates = release_users.take(positions[0]).astype(int).tolist()
        rows.append({
            "target_user": target_user,
            "top_candidate": candidates[0],
            "top1_hit": candidates[0] == target_user,
            "topk_hit": target_user in candidates,
            "nearest_cosine_similarity": 1 - float(distances[0][0]),
        })
    return pd.DataFrame(rows)

positive_profiles = sample_profiles(release, seed=SEED)
negative_profiles = sample_profiles(holdout, seed=SEED + 1)
nn_results = nearest_neighbor_attack(positive_profiles)
nn_results[["top1_hit", "topk_hit", "nearest_cosine_similarity"]].mean()

## Logistic membership inference

Use nearest-neighbor score features, then fit scikit-learn `LogisticRegression` to classify whether the external profile came from a release user.

In [ ]:
def membership_features(profiles: dict[int, pd.DataFrame]) -> pd.DataFrame:
    rows = []
    for _user, profile in profiles.items():
        vector = profile_vector(profile, release_movies)
        distances, _positions = nn.kneighbors(vector, n_neighbors=5)
        sims = 1 - distances[0]
        rows.append({
            "best_similarity": float(sims[0]),
            "similarity_gap": float(sims[0] - sims[1]),
            "mean_top5_similarity": float(sims.mean()),
            "known_movies_in_release": int(vector.nnz),
        })
    return pd.DataFrame(rows)

X = pd.concat([membership_features(positive_profiles), membership_features(negative_profiles)], ignore_index=True)
y = np.r_[np.ones(len(positive_profiles)), np.zeros(len(negative_profiles))]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=SEED)
clf = LogisticRegression(max_iter=1_000).fit(X_train, y_train)
proba = clf.predict_proba(X_test)[:, 1]
{"accuracy": accuracy_score(y_test, proba >= 0.5), "auc": roc_auc_score(y_test, proba)}

In [ ]:
pd.Series(clf.coef_[0], index=X.columns).sort_values(key=np.abs, ascending=False)

## Compare releases

The same scikit-learn attack can audit each anonymized release. Releases that reduce AUC/top-k hit rate while preserving utility are better blue-team candidates.

In [ ]:
def release_membership_auc(release_frame: pd.DataFrame, rating_col: str) -> float:
    frame = release_frame[["movie_id", "customer_id", rating_col]].rename(columns={rating_col: "rating"}).dropna()
    global release_users, release_movies, release_matrix, nn
    release_users, release_movies, release_matrix = fit_user_vectorizer(frame)
    nn = NearestNeighbors(metric="cosine", algorithm="brute", n_neighbors=5).fit(release_matrix)
    X_rel = pd.concat([membership_features(positive_profiles), membership_features(negative_profiles)], ignore_index=True)
    X_train, X_test, y_train, y_test = train_test_split(X_rel, y, test_size=0.3, stratify=y, random_state=SEED)
    model = LogisticRegression(max_iter=1_000).fit(X_train, y_train)
    return roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])

rows = []
for rel in make_releases(release, rare_movie_min_users=100, k_suppression=5):
    if rel.rating_col is not None:
        rows.append({"release_name": rel.name, "membership_auc": release_membership_auc(rel.data, rel.rating_col)})
pd.DataFrame(rows).sort_values("membership_auc", ascending=False)